In [0]:
%sql
USE CATALOG retail_dwh;

In [0]:
%sql
    
CREATE TABLE IF NOT EXISTS retail_dwh.gold.DimCustomer
(
    CustomerSK      BIGINT GENERATED ALWAYS AS IDENTITY,
    CustomerID      STRING,
    CustomerName    STRING,
    Email           STRING,
    City            STRING,
    Address         STRING,
    StartDate       DATE,
    EndDate         DATE,
    IsActive        INT
)
USING DELTA
LOCATION 's3a://retail-sales-dwh-sadhvika/gold/DimCustomer';

In [0]:
%sql
-- Expire old records where City OR Address has changed
MERGE INTO retail_dwh.gold.DimCustomer AS target
USING (
    SELECT
        TRIM(CustomerID)            AS CustomerID,
        INITCAP(TRIM(CustomerName)) AS CustomerName,
        LOWER(TRIM(Email))          AS Email,
        TRIM(City)                  AS City,
        TRIM(Address)               AS Address
    FROM retail_dwh.silver.customers
) AS source
ON  target.CustomerID = source.CustomerID
AND target.IsActive   = 1
WHEN MATCHED AND (
    target.City    != source.City   OR
    target.Address != source.Address
)
THEN UPDATE SET
    target.IsActive = 0,
    target.EndDate  = CURRENT_DATE();


In [0]:
%sql
-- Insert new active records for:
-- 1. Brand new customers
-- 2. Existing customers whose City or Address changed
INSERT INTO retail_dwh.gold.DimCustomer
(
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
)
SELECT DISTINCT
    TRIM(s.CustomerID)              AS CustomerID,
    INITCAP(TRIM(s.CustomerName))   AS CustomerName,
    LOWER(TRIM(s.Email))            AS Email,
    TRIM(s.City)                    AS City,
    TRIM(s.Address)                 AS Address,
    CURRENT_DATE()                  AS StartDate,
    DATE('9999-12-31')              AS EndDate,
    1                               AS IsActive
FROM retail_dwh.silver.customers s
WHERE NOT EXISTS (
    SELECT 1
    FROM retail_dwh.gold.DimCustomer d
    WHERE d.CustomerID = TRIM(s.CustomerID)
      AND d.City       = TRIM(s.City)
      AND d.Address    = TRIM(s.Address)
      AND d.IsActive   = 1
);

In [0]:
%sql
-- Scenario 1: Same customer no change → should have only 1 active record
-- Scenario 2: City changed → old expired, new active
-- Scenario 3: Address changed → old expired, new active
-- Scenario 4: Duplicate → only 1 active record
SELECT
    CustomerID,
    CustomerName,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
FROM retail_dwh.gold.DimCustomer
ORDER BY CustomerID, StartDate;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_dwh.gold.DimProduct
(
    ProductSK       BIGINT GENERATED ALWAYS AS IDENTITY,
    ProductID       STRING,
    ProductName     STRING,
    Category        STRING,
    UnitPrice       DECIMAL(10,2),
    EffectiveDate   DATE
)
USING DELTA
LOCATION 's3a://retail-sales-dwh-sadhvika/gold/DimProduct';

In [0]:
%sql
-- Trim ProductName and Category 
MERGE INTO retail_dwh.gold.DimProduct AS target
USING (
    SELECT
        TRIM(ProductID)                  AS ProductID,
        TRIM(ProductName)                AS ProductName,
        TRIM(Category)                   AS Category,
        CAST(UnitPrice AS DECIMAL(10,2)) AS UnitPrice,
        CURRENT_DATE()                   AS EffectiveDate
    FROM retail_dwh.silver.products
) AS source
ON target.ProductID = source.ProductID
WHEN MATCHED THEN UPDATE SET
    target.ProductName    = source.ProductName,
    target.Category       = source.Category,
    target.UnitPrice      = source.UnitPrice,
    target.EffectiveDate  = source.EffectiveDate
WHEN NOT MATCHED THEN INSERT
(
    ProductID,
    ProductName,
    Category,
    UnitPrice,
    EffectiveDate
)
VALUES
(
    source.ProductID,
    source.ProductName,
    source.Category,
    source.UnitPrice,
    source.EffectiveDate
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_dwh.gold.DimStore
(
    StoreSK     BIGINT GENERATED ALWAYS AS IDENTITY,
    StoreID     STRING,
    StoreName   STRING,
    Region      STRING
)
USING DELTA
LOCATION 's3a://retail-sales-dwh-sadhvika/gold/DimStore';

In [0]:
%sql
-- Trim StoreName and Region as per doc
MERGE INTO retail_dwh.gold.DimStore AS target
USING (
    SELECT
        TRIM(StoreID)   AS StoreID,
        TRIM(StoreName) AS StoreName,
        TRIM(Region)    AS Region
    FROM retail_dwh.silver.stores
) AS source
ON target.StoreID = source.StoreID
WHEN MATCHED THEN UPDATE SET
    target.StoreName = source.StoreName,
    target.Region    = source.Region
WHEN NOT MATCHED THEN INSERT
(
    StoreID,
    StoreName,
    Region
)
VALUES
(
    source.StoreID,
    source.StoreName,
    source.Region
);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail_dwh.gold.FactSales
(
    SalesSK         BIGINT GENERATED ALWAYS AS IDENTITY,
    TransactionID   STRING,
    CustomerSK      INT,
    ProductSK       INT,
    StoreSK         INT,
    Quantity        INT,
    Amount          DECIMAL(10,2),
    TxnDate         DATE
)
USING DELTA
LOCATION 's3a://retail-sales-dwh-sadhvika/gold/FactSales';

In [0]:
%sql
-- Lookup ACTIVE CustomerSK only (IsActive = 1) as per doc
-- Derive Amount = Quantity x UnitPrice
-- Standardize TxnDate to YYYY-MM-DD
MERGE INTO retail_dwh.gold.FactSales AS target
USING (
    SELECT
        TRIM(st.TransactionID)                              AS TransactionID,
        dc.CustomerSK,
        dp.ProductSK,
        ds.StoreSK,
        CAST(st.Quantity AS INT)                            AS Quantity,
        CAST(st.Quantity * dp.UnitPrice AS DECIMAL(10,2))   AS Amount,
        CAST(st.TxnDate AS DATE)                            AS TxnDate
    FROM retail_dwh.silver.sales_transactions st
    -- Active CustomerSK lookup only
    JOIN retail_dwh.gold.DimCustomer dc
        ON  TRIM(st.CustomerID) = dc.CustomerID
        AND dc.IsActive         = 1
    -- ProductSK lookup
    JOIN retail_dwh.gold.DimProduct dp
        ON TRIM(st.ProductID)   = dp.ProductID
    -- StoreSK lookup
    JOIN retail_dwh.gold.DimStore ds
        ON TRIM(st.StoreID)     = ds.StoreID
) AS source
ON target.TransactionID = source.TransactionID
WHEN MATCHED THEN UPDATE SET
    target.CustomerSK = source.CustomerSK,
    target.ProductSK  = source.ProductSK,
    target.StoreSK    = source.StoreSK,
    target.Quantity   = source.Quantity,
    target.Amount     = source.Amount,
    target.TxnDate    = source.TxnDate
WHEN NOT MATCHED THEN INSERT
(
    TransactionID,
    CustomerSK,
    ProductSK,
    StoreSK,
    Quantity,
    Amount,
    TxnDate
)
VALUES
(
    source.TransactionID,
    source.CustomerSK,
    source.ProductSK,
    source.StoreSK,
    source.Quantity,
    source.Amount,
    source.TxnDate
);